In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 16.0,
    'font.family': 'serif',
    'font.serif': 'Palatino',
    'axes.titlesize': 'medium',
    'figure.titlesize': 'large',
    'legend.fontsize': 'medium',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'figure.autolayout': True,
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{siunitx}[=v2]",
})

from pathlib import Path

PLOT_ROOT = Path.cwd() / "plots"
PLOT_ROOT.mkdir(exist_ok=True)

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.io as sio

sys.path.insert(0, str(Path.cwd()))
from arsw_python import run_optimepsilon_TD86

# ── Repository tree ────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent
ARSW_TOOLKIT  = REPO_ROOT / "ARSW2015" / "ARSW2015-toolkit"

# ── Data inputs ────────────────────────────────────────────────────────────────
MAT_PATH_TD86  = ARSW_TOOLKIT / "matlab" / "data" / "input" / "prepdata_big_TD86.mat"
CSV_PATH_WAGES = ARSW_TOOLKIT / "matlab" / "data" / "input" / "wageworker1986.csv"
TTM_MAT_PATH   = Path.cwd() / "TTM" / "travel_time_matrices.mat"

# ── Shapefiles ─────────────────────────────────────────────────────────────────
# Berlin4matlab.shp: full Berlin, same file used in TTM/Final.py (ARSW toolkit)
# WestBerlin4matlab.shp: West Berlin subset, block ordering == prepdata_big_TD86.mat
BLOCKS_SHP_FULL = ARSW_TOOLKIT / "shapefile" / "Berlin4matlab.shp"
BLOCKS_SHP_WEST = ARSW_TOOLKIT / "shapefile" / "WestBerlin4matlab.shp"

# ── Parameters ─────────────────────────────────────────────────────────────────
# κε = ν̂ estimated in Task 1(a) via gravity regression (ARSW eq. 23/25).
# Update KAPPAEPS to the Column (2) OLS estimate from your task_1a output.
# ARSW (2015) benchmark value (Table III, Col. 2): 0.07.
KAPPAEPS = 0.07   # ← replace with your Task 1(a) ν̂

ALPHA = 0.80   # labour share in production (ARSW calibration)
BETA  = 0.75   # non-housing expenditure share (ARSW calibration)

## Task 1(b) — Estimating ε and κ via One-Step GMM

**Setup.** Given $\hat{\nu} = \varepsilon\kappa$ from Task 1(a), we disentangle $\varepsilon$
and $\kappa$ using within-city wage dispersion as an additional moment.

**Transformed wage system (ARSW eq. S.44).** For fixed $\kappa\varepsilon$ and given
travel times $\tau_{ij}$, the vector $\omega \equiv w^\varepsilon$ satisfies the
fixed-point equation

$$
\omega_j \propto \left(\frac{H_j^M}{\sum_i H_i^R \cdot \frac{\omega_j}{\sum_k \omega_k / c_{kj}}}\right)
$$

where $c_{ij} = e^{\kappa\varepsilon \cdot \tau_{ij}}$ are commuting iceberg costs.
This is solved once by `solve_omega_O` (≡ `comegaoptO.m`) for fixed $\kappa\varepsilon$,
independently of $\varepsilon$.

**GMM moment condition (ARSW eq. S.64).** Adjusted wages satisfy $w_j = \omega_j^{1/\varepsilon}$.
The Fréchet parameter $\varepsilon$ governs the dispersion of wages across blocks:

$$
\hat{\varepsilon} = \arg\min_\varepsilon \left[\,\widehat{\mathrm{Var}}_J(\ln w_J^{\text{model}}) - \widehat{\mathrm{Var}}_J(\ln w_J^{\text{data}})\,\right]^2
$$

where $w_J^{\text{model}}$ are Bezirk-level employment-weighted wages computed from
$\omega^{1/\varepsilon}$, and $w_J^{\text{data}}$ are the observed 1986 wages from
`wageworker1986.csv`. Since $\omega$ is pre-solved and fixed, the objective depends
on $\varepsilon$ only through the $1/\varepsilon$ exponent applied to $\omega$.

**Identification.** Higher $\varepsilon$ compresses the wage distribution (workers are
more homogeneous), lower $\varepsilon$ amplifies it. The unique $\hat{\varepsilon}$ matches
model wage dispersion to the empirical cross-Bezirk log-wage variance.

**Deriving κ.** Once $\hat{\varepsilon}$ is known:
$$\hat{\kappa} = \frac{\hat{\nu}}{\hat{\varepsilon}} = \frac{\kappa\varepsilon}{\varepsilon}$$

**West Berlin 1986.** The estimation uses West Berlin blocks only (`rw` suffix in
MATLAB variables). The sample covers the period before reunification, when West
Berlin was an economically isolated city — providing a clean monocentric-city setting.

In [ ]:
# ── Check required inputs ──────────────────────────────────────────────────────

missing = []

if not MAT_PATH_TD86.exists():
    missing.append(MAT_PATH_TD86)
    print("⚠  prepdata_big_TD86.mat NOT FOUND.")
    print(f"   Expected: {MAT_PATH_TD86}")
    print()
    print("   Download from:")
    print("   https://box.hu-berlin.de/f/3f364937b4a240e883f8/?dl=1")
    print(f"   Save to: {MAT_PATH_TD86.parent}")
else:
    print(f"✓  prepdata_big_TD86.mat  ({MAT_PATH_TD86.stat().st_size / 1e6:.1f} MB)")

if not CSV_PATH_WAGES.exists():
    missing.append(CSV_PATH_WAGES)
    print(f"⚠  wageworker1986.csv NOT FOUND: {CSV_PATH_WAGES}")
else:
    print(f"✓  wageworker1986.csv")

USE_USER_TTM = TTM_MAT_PATH.exists()
if not USE_USER_TTM:
    print(f"⚠  travel_time_matrices.mat not found at {TTM_MAT_PATH}.")
    print("   Estimation will use the ARSW-provided tt86rw from prepdata_big_TD86.mat.")
else:
    print(f"✓  travel_time_matrices.mat  ({TTM_MAT_PATH.stat().st_size / 1e6:.1f} MB)")

if missing:
    raise FileNotFoundError(
        f"Required files missing: {missing}\n"
        "Cannot proceed without prepdata_big_TD86.mat and wageworker1986.csv."
    )

# ── Preview prepdata_big_TD86 structure ────────────────────────────────────────
_raw = sio.loadmat(str(MAT_PATH_TD86), squeeze_me=True, struct_as_record=False)
_keys = [k for k in _raw.keys() if not k.startswith("_")]
_nobs = int(_raw["nobs86rw"])
_tt_shape = _raw["tt86rw"].shape

print(f"\nprepdata_big_TD86 keys : {_keys}")
print(f"nobs86rw (West Berlin) : {_nobs} blocks")
print(f"tt86rw shape           : {_tt_shape}")
print(f"empwpl86rw sum         : {_raw['empwpl86rw'].sum():,.0f} (total workplace employment)")
print(f"bzk86rw unique values  : {sorted(set(_raw['bzk86rw'].astype(int).tolist()))}")
del _raw

In [ ]:
def _clean_shapefile(gdf: gpd.GeoDataFrame, target_crs: int = 25833) -> gpd.GeoDataFrame:
    """
    Six-step geometry cleaning identical to TTM/Final.py.
    Must be applied to BOTH shapefiles to ensure centroid_id consistency.
    """
    def _valid_coords(geom):
        try:
            b = geom.bounds
            return len(b) == 4 and not (np.any(np.isnan(b)) or np.any(np.isinf(b)))
        except Exception:
            return False

    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].reset_index(drop=True)
    gdf = gdf[gdf.geometry.apply(_valid_coords)].reset_index(drop=True)
    gdf["geometry"] = gdf.geometry.make_valid()
    gdf = gdf.to_crs(epsg=target_crs)
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].reset_index(drop=True)
    gdf = gdf[gdf.geometry.apply(_valid_coords)].reset_index(drop=True)
    return gdf


def build_west_berlin_ttm(
    full_ttm_mat_path: Path,
    blocks_shp_full: Path,
    blocks_shp_west: Path,
    mat_key: str = "tt_matrix_full",
    nan_fill: float = 9999.0,
    verbose: bool = True,
) -> tuple[np.ndarray, list]:
    """
    Extract the West Berlin sub-matrix from the user's full-Berlin travel time matrix.

    The returned matrix's block ordering matches WestBerlin4matlab.shp after the
    same geometry cleaning applied in Final.py, which equals the ordering of
    workplace/residence vectors in prepdata_big_TD86.mat.

    Parameters
    ----------
    full_ttm_mat_path : path to travel_time_matrices.mat (Topic_7/TTM/)
    blocks_shp_full   : Berlin4matlab.shp (ARSW toolkit) — same as Final.py
    blocks_shp_west   : WestBerlin4matlab.shp (ARSW toolkit)
    mat_key           : 'tt_matrix_full' or 'tt_matrix_simple'
    nan_fill          : replace NaN/inf travel times (disconnected pairs) with this

    Returns
    -------
    west_tt   : (nobs86rw, nobs86rw) float64 ndarray
    west_cids : list of centroid_ids corresponding to West Berlin blocks,
                in WestBerlin4matlab order
    """
    # 1. Load + clean full Berlin shapefile; assign centroid_ids
    if verbose:
        print("  [1/4] Cleaning Berlin4matlab.shp ...")
    gdf_full = _clean_shapefile(gpd.read_file(str(blocks_shp_full)))
    gdf_full["centroid_id"] = [f"centroid_{i}" for i in range(len(gdf_full))]
    centroids_full = gpd.GeoDataFrame(
        {"centroid_id": gdf_full["centroid_id"].values},
        geometry=gdf_full.geometry.centroid,
        crs=25833,
    )
    if verbose:
        print(f"      Full Berlin: {len(gdf_full)} blocks")

    # 2. Load + clean West Berlin shapefile
    if verbose:
        print("  [2/4] Cleaning WestBerlin4matlab.shp ...")
    gdf_west = _clean_shapefile(gpd.read_file(str(blocks_shp_west)))
    centroids_west = gpd.GeoDataFrame(
        {"west_idx": range(len(gdf_west))},
        geometry=gdf_west.geometry.centroid,
        crs=25833,
    )
    if verbose:
        print(f"      West Berlin: {len(gdf_west)} blocks")

    # 3. Nearest-neighbour match: West Berlin centroid → Berlin4matlab centroid_id
    if verbose:
        print("  [3/4] Matching West Berlin blocks to full-Berlin centroid_ids ...")
    matched = gpd.sjoin_nearest(
        centroids_west,
        centroids_full,
        how="left",
        distance_col="match_dist_m",
    )
    # Resolve ties: keep minimum-distance match per West Berlin block
    matched = matched.sort_values("match_dist_m").groupby("west_idx").first().reset_index()
    matched = matched.sort_values("west_idx")   # restore original West Berlin ordering

    max_dist = matched["match_dist_m"].max()
    if verbose:
        print(f"      Max centroid match distance: {max_dist:.3f} m")
    if max_dist > 10.0:
        print(f"      ⚠  Warning: max match distance = {max_dist:.1f} m. "
              "Shapefiles may not be perfectly aligned.")

    west_cids = matched["centroid_id"].tolist()
    if len(set(west_cids)) != len(west_cids):
        n_dup = len(west_cids) - len(set(west_cids))
        raise ValueError(
            f"{n_dup} duplicate centroid_id matches found. "
            "Two West Berlin blocks mapped to the same full-Berlin block — "
            "inspect shapefile geometry."
        )

    # 4. Load user TTM, extract sub-matrix
    if verbose:
        print(f"  [4/4] Loading {full_ttm_mat_path.name} and extracting sub-matrix ...")
    raw_ttm = sio.loadmat(str(full_ttm_mat_path), squeeze_me=True)
    centroid_ids_mat = list(raw_ttm["centroid_ids"])
    id_to_idx = {cid: i for i, cid in enumerate(centroid_ids_mat)}
    if verbose:
        print(f"      Mat centroid_ids: {len(centroid_ids_mat)}")

    missing_cids = [cid for cid in west_cids if cid not in id_to_idx]
    if missing_cids:
        raise ValueError(
            f"{len(missing_cids)} West Berlin centroid_ids not found in TTM mat. "
            f"First few: {missing_cids[:5]}. "
            "Ensure travel_time_matrices.mat was built from the same Berlin4matlab.shp."
        )

    west_indices = np.array([id_to_idx[cid] for cid in west_cids])

    tt_full = raw_ttm[mat_key].astype(np.float64)
    west_tt = tt_full[np.ix_(west_indices, west_indices)]

    # 5. Handle disconnected pairs: NaN / inf → nan_fill minutes
    n_nan = int(np.isnan(west_tt).sum() + np.isinf(west_tt).sum())
    if n_nan > 0 and verbose:
        pct = 100 * n_nan / west_tt.size
        print(f"      Replacing {n_nan} NaN/inf entries ({pct:.1f}%) with {nan_fill} min.")
    west_tt = np.where(np.isnan(west_tt) | np.isinf(west_tt), nan_fill, west_tt)
    np.fill_diagonal(west_tt, 0.0)   # self-travel = 0

    off_diag = west_tt[west_tt > 0]
    finite_off_diag = off_diag[off_diag < nan_fill]
    if verbose and len(finite_off_diag) > 0:
        print(f"      Off-diagonal travel time (connected pairs): "
              f"[{finite_off_diag.min():.1f}, {finite_off_diag.max():.1f}] min, "
              f"mean = {finite_off_diag.mean():.1f} min")

    return west_tt, west_cids


# ── Execute alignment ──────────────────────────────────────────────────────────
tt86rw_override = None

if USE_USER_TTM:
    print("=== Aligning user TTM to West Berlin block ordering ===")
    try:
        tt86rw_override, west_berlin_cids = build_west_berlin_ttm(
            TTM_MAT_PATH,
            BLOCKS_SHP_FULL,
            BLOCKS_SHP_WEST,
            mat_key="tt_matrix_full",    # use full multi-modal network
            verbose=True,
        )
        # Validate shape against prepdata_big_TD86
        _nobs_check = int(sio.loadmat(str(MAT_PATH_TD86), squeeze_me=True)["nobs86rw"])
        if tt86rw_override.shape[0] != _nobs_check:
            print(f"\n⚠  Shape mismatch: user TTM West Berlin = {tt86rw_override.shape[0]}, "
                  f"prepdata_big_TD86 nobs86rw = {_nobs_check}.")
            print("   Falling back to default ARSW-provided tt86rw.")
            tt86rw_override = None
        else:
            print(f"\n✓  Override matrix shape: {tt86rw_override.shape}  — matches nobs86rw")
    except Exception as e:
        print(f"\n⚠  TTM alignment failed: {e}")
        print("   Falling back to default ARSW-provided tt86rw.")
        tt86rw_override = None
else:
    west_berlin_cids = None

print(f"\ntt86rw source: {'user TTM (travel_time_matrices.mat)' if tt86rw_override is not None else 'ARSW prepdata_big_TD86.mat (default)'}")

In [ ]:
print("=" * 72)
print("  TASK 1(b) — One-Step GMM: Estimating ε via wage dispersion moment")
print("=" * 72)
print(f"  κε = ν̂ = {KAPPAEPS}  (from Task 1a)")
print(f"  Using user TTM override: {tt86rw_override is not None}")
print(f"  Search range: ε ∈ [{2.0}, {24.0}]")
print()

results_1b = run_optimepsilon_TD86(
    mat_path_TD86=MAT_PATH_TD86,
    csv_path_wages=CSV_PATH_WAGES,
    tt86rw_override=tt86rw_override,
    alpha=ALPHA,
    beta=BETA,
    kappaeps=KAPPAEPS,
    epsilon_lb=2.0,
    epsilon_ub=24.0,
    verbose=True,
)

epsilon_hat = results_1b["epsilon"]
kappa_hat   = results_1b["kappa"]

print(f"\n  → ε̂ = {epsilon_hat:.2f}")
print(f"  → κ̂ = {kappa_hat:.6f}")

In [ ]:
# ── Formatted results ──────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════╗")
print("║  Task 1(b) — Results                                            ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  ε̂  (Fréchet dispersion)         {epsilon_hat:8.2f}                   ║")
print(f"║  κ̂  (commuting iceberg decay)    {kappa_hat:10.6f}                 ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  GMM moment comparison:                                         ║")
print(f"║    Var(ln w, data / Bezirke) =  {results_1b['varlwdata']:8.6f}                ║")
print(f"║    Var(ln w, model / blocks) =  {results_1b['varlwage86rw']:8.6f}                ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  ARSW (2015) benchmark:  ε = 6.83,  κ = 0.0102  (Table V)     ║")
print(f"║  Input:  κε = ν̂ = {KAPPAEPS}  (Task 1a estimate)               ║")
print("╚══════════════════════════════════════════════════════════════════╝")

# LaTeX-ready parameter output
print("\n--- LaTeX parameters ---")
print(f"\\hat{{\\varepsilon}} = {epsilon_hat:.2f}, \\quad \\hat{{\\kappa}} = {kappa_hat:.4f}")

In [ ]:
# ── Load West Berlin shapefile for mapping ─────────────────────────────────────
print("Loading WestBerlin4matlab.shp for visualization ...")
gdf_west_map = _clean_shapefile(gpd.read_file(str(BLOCKS_SHP_WEST)))
n_shp  = len(gdf_west_map)
n_vec  = len(results_1b["omega86rw"])

if n_shp != n_vec:
    print(f"⚠  Shapefile blocks ({n_shp}) ≠ ω vector length ({n_vec}).")
    print("   Maps will use the first min(n_shp, n_vec) entries — verify alignment.")
    n_plot = min(n_shp, n_vec)
    gdf_plot = gdf_west_map.iloc[:n_plot].copy().reset_index(drop=True)
else:
    n_plot = n_shp
    gdf_plot = gdf_west_map.copy()

omega_vec = results_1b["omega86rw"][:n_plot]
wage_vec  = results_1b["wage86rw"][:n_plot]

# Assign; zero-employment blocks (ω = 0) → NaN for clean masking
gdf_plot["log_omega"] = np.where(omega_vec > 0, np.log(omega_vec), np.nan)
gdf_plot["adj_wage"]  = np.where(wage_vec  > 0, wage_vec, np.nan)

# ── Two-panel figure ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: transformed wages ω
gdf_plot.plot(
    column="log_omega",
    ax=axes[0],
    cmap="YlOrRd",
    legend=True,
    missing_kwds={"color": "#d0d0d0", "label": "No employment"},
    legend_kwds={
        "label": r"$\log\,\omega_j$",
        "shrink": 0.70,
        "orientation": "vertical",
    },
)
axes[0].set_title(
    r"Transformed wages $\log\,\omega_j$" "\n" r"(West Berlin, 1986; $\omega_j = w_j^{\varepsilon}$)",
    pad=8,
)
axes[0].set_axis_off()

# Panel B: adjusted wages w (normalized to geometric mean = 1)
gdf_plot.plot(
    column="adj_wage",
    ax=axes[1],
    cmap="YlOrRd",
    legend=True,
    missing_kwds={"color": "#d0d0d0", "label": "No employment"},
    legend_kwds={
        "label": r"$w_j\,/\,\bar{w}_{\mathrm{gm}}$",
        "shrink": 0.70,
        "orientation": "vertical",
    },
)
axes[1].set_title(
    r"Adjusted wages $w_j\,/\,\bar{w}_{\mathrm{gm}}$" "\n" r"(West Berlin, 1986; normalized by geometric mean)",
    pad=8,
)
axes[1].set_axis_off()

plt.suptitle(
    rf"Task 1(b): West Berlin 1986 — $\hat{{\varepsilon}} = {epsilon_hat:.2f}$, "
    rf"$\hat{{\kappa}} = {kappa_hat:.4f}$",
    fontsize=14,
    y=1.01,
)
plt.tight_layout()

save_path = PLOT_ROOT / "task_1b_wage_maps.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"\nSaved: {save_path}")

# ── Wage summary statistics ───────────────────────────────────────────────────
pos = wage_vec > 0
print(f"\nAdjusted wages (positive blocks only, N={pos.sum()}):")
print(f"  Min:    {wage_vec[pos].min():.4f}")
print(f"  Median: {np.median(wage_vec[pos]):.4f}")
print(f"  Max:    {wage_vec[pos].max():.4f}")
print(f"  Geomean (should be ≈ 1.0): {np.exp(np.log(wage_vec[pos]).mean()):.6f}")

In [ ]:
print("══════════════════════════════════════════════════════════════════════")
print("  Task 1(b) — Summary")
print("══════════════════════════════════════════════════════════════════════")
print()
print("  Estimation strategy:")
print("  ─────────────────────────────────────────────────────────────────")
print("  1. Load prepdata_big_TD86: floor prices, employment, travel times")
print(f"     West Berlin 1986: {int(sio.loadmat(str(MAT_PATH_TD86), squeeze_me=True)['nobs86rw'])} blocks")
print()
print("  2. Solve transformed wage system for fixed κε (comegaoptO):")
print(f"     ω satisfies Eq. S.44 with κε = {KAPPAEPS}")
print(f"     Converged: {True}  (gap reported above)")
print()
print("  3. One-step GMM (patternsearch → bounded Brent's method):")
print(f"     Minimize [ Var_J(ln w_J^model) − Var_J(ln w_J^data) ]²")
print()
print(f"  ┌────────────────────────────────────────────────────────────┐")
print(f"  │  ε̂ = {epsilon_hat:.2f}   (ARSW 2015 benchmark: 6.83)              │")
print(f"  │  κ̂ = {KAPPAEPS} / {epsilon_hat:.2f} = {kappa_hat:.6f}                 │")
print(f"  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Var(ln w, data):    {results_1b['varlwdata']:.6f}                     │")
print(f"  │  Var(ln w, blocks):  {results_1b['varlwage86rw']:.6f}                     │")
print(f"  └────────────────────────────────────────────────────────────┘")
print()
print("  Interpretation:")
print(f"  ε̂ = {epsilon_hat:.2f} governs the cross-sectional dispersion of wages.")
print(f"  Lower ε → stronger sorting, higher wage inequality.")
print(f"  κ̂ = {kappa_hat:.6f}: each extra minute of commuting reduces utility by")
print(f"  the factor exp(−κ) (iceberg commuting cost).")
print()
print("  → These estimates feed directly into Task 1(c): recovery of")
print("    fundamental productivities A and amenities B for all Berlin")
print("    blocks in 2006 via run_calcal_TD().")
print("══════════════════════════════════════════════════════════════════════")